In [ ]:
#Deps (SmolLM3 needs a recent transformers; datasets pulls Tobi-Bueck)
!pip -q install -U "transformers>=4.53" "trl>=0.9" peft bitsandbytes datasets accelerate
!pip -q uninstall -y torchao   # Kaggle ships an old torchao that recent peft rejects; we use bitsandbytes
print("deps installed")

In [ ]:
#Config + helpers. SAME pipeline as pipeline2; only base_model + the task data differ.
import os, json, re, random
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
TASK   = "/kaggle/working/data"                                  # task data, built in the next cell
PROBES = "/kaggle/input/datasets/ianrusseladem/pipeline2-data"   # reuse the general-ability probes + rehearsal
OUT    = "/kaggle/working"

CFG = {
    "name": "support-ticket-queue",
    "base_model": "HuggingFaceTB/SmolLM3-3B",   # swap to Qwen/Qwen2.5-0.5B-Instruct to retarget, no code change
    "lora": {"rank": 16, "alpha": 32, "dropout": 0.05, "target_modules": "all-linear"},
    "train": {"epochs": 6, "lr": 2e-4, "max_len": 1024, "batch": 1, "grad_accum": 16,
              "seeds": [0, 1],   # decide on the MEDIAN across seeds, robust to run-to-run noise
              # adaptive length ON. NOTE: 3B + per-epoch clean-eval loads a 2nd bf16 copy; if it OOMs
              # on a T4, set enabled: False, or validate the loop on Qwen (0.5B) first.
              "early_stopping": {"enabled": True, "patience": 2, "min_task_delta": 0.005,
                                 "regression_tolerance": 0.05, "eval_task_limit": 0, "eval_batch": 8}},
    "guardrails": {"replay_mix": True},
    "acceptance": {"min_task_gain_macro_f1": 0.05, "max_regression_drop": 0.05},
    "adjust_on_fail": {"force_replay": True, "lower_lr_factor": 0.5, "reduce_epochs_to": 2},
}
LABELS = []   # filled by the data-prep cell

def read_jsonl(p): return [json.loads(l) for l in open(p) if l.strip()]
def normalize(s): return " ".join(str(s).lower().split())
def c_predict_label(out, labels):
    tail = out.rsplit("</think>", 1)[-1] if "</think>" in out else out
    by = {normalize(l): l for l in labels}
    lines = [x for x in tail.splitlines() if x.strip()]
    last = normalize(lines[-1]) if lines else ""
    if last in by: return by[last]
    low = normalize(tail); hits = [l for l in labels if normalize(l) in low]
    return max(hits, key=len) if hits else None
def macro_f1(gold, pred, labels):
    tot = 0.0
    for l in labels:
        tp = sum(1 for g,p in zip(gold,pred) if g==l and p==l)
        fp = sum(1 for g,p in zip(gold,pred) if g!=l and p==l)
        fn = sum(1 for g,p in zip(gold,pred) if g==l and p!=l)
        pr = tp/(tp+fp) if tp+fp else 0.0; rc = tp/(tp+fn) if tp+fn else 0.0
        tot += 2*pr*rc/(pr+rc) if pr+rc else 0.0
    return tot/len(labels)
def score_probes(raw, probes, mode):
    ok = 0
    for o, p in zip(raw, probes):
        want = [a.lower() for a in p["answers"]]; low = o.lower()
        ok += (all(a in low for a in want) if mode=="all" else any(a in low for a in want))
    return ok/len(probes) if probes else 0.0
print("config ready")

In [ ]:
#Build the queue-classification dataset from Tobi-Bueck (the only task-specific code)
from datasets import load_dataset
from collections import Counter, defaultdict

LABEL_FIELD_CANDIDATES = ["queue", "type", "priority", "category"]
TEXT_FIELDS = ["subject", "body"]; LANG_FIELD, KEEP_LANG = "language", "en"
GOLD_PER_LABEL, SEED_PER_LABEL = 20, 40; MIN_LABEL_COUNT = GOLD_PER_LABEL + SEED_PER_LABEL
REPLAY_FRACTION = 0.25
INSTR = ("You are a support-ticket router. Read the ticket and reply with exactly one queue label "
         "from the list below, copied verbatim, with nothing else.\nQueues:\n{labels}")

def norm(s): return " ".join(str(s).lower().split())
def ticket_text(r): return "\n".join(str(r.get(f,"")).strip() for f in TEXT_FIELDS if r.get(f))
def msgs(text, labels, label=None):
    m = [{"role":"system","content":INSTR.format(labels="\n".join(labels))},
         {"role":"user","content":text}]
    if label is not None: m.append({"role":"assistant","content":label})
    return m

random.seed(0); os.makedirs(TASK, exist_ok=True)
ds = load_dataset("Tobi-Bueck/customer-support-tickets", split="train")
cols = ds.column_names; print("columns:", cols)
label_field = next(f for f in LABEL_FIELD_CANDIDATES if f in cols)
print("label field:", label_field)

rows = []
for r in ds:
    if LANG_FIELD in cols and norm(r.get(LANG_FIELD)) not in ("", KEEP_LANG, "english"): continue
    t, l = ticket_text(r), r.get(label_field)
    if t and l and str(l).strip(): rows.append((t, str(l).strip()))
counts = Counter(l for _,l in rows)
labels = sorted(l for l,c in counts.items() if c >= MIN_LABEL_COUNT)
assert labels, f"no label has >= {MIN_LABEL_COUNT} rows; counts={dict(counts)}"
print(f"usable rows={len(rows)}, kept labels ({len(labels)}): {labels}")

by = defaultdict(list)
for t,l in rows:
    if l in labels: by[l].append(t)
gold, seed_rows, seen = [], [], set()
for l in labels:
    pool = by[l][:]; random.shuffle(pool)
    uniq, local = [], set()
    for t in pool:
        k = norm(t)
        if k not in local: local.add(k); uniq.append(t)
    for t in uniq[:GOLD_PER_LABEL]: gold.append((t,l)); seen.add(norm(t))
    for t in uniq[GOLD_PER_LABEL:GOLD_PER_LABEL+SEED_PER_LABEL]: seed_rows.append((t,l))
seed_rows = [(t,l) for t,l in seed_rows if norm(t) not in seen]; random.shuffle(seed_rows)

LABELS = labels   # set the global the helpers use
open(f"{TASK}/labels.txt","w").write("\n".join(labels)+"\n")
with open(f"{TASK}/gold.jsonl","w") as f:
    for t,l in gold: f.write(json.dumps({"messages":msgs(t,labels),"label":l})+"\n")
with open(f"{TASK}/train_synth.jsonl","w") as f:
    for t,l in seed_rows: f.write(json.dumps({"messages":msgs(t,labels,l)})+"\n")

# train_mix = task + a general rehearsal sample (reuse pipeline2-data's train_synth as the pool)
mix = [msgs(t,labels,l) for t,l in seed_rows]; reh = []
pool_path = f"{PROBES}/train_synth.jsonl"
if os.path.exists(pool_path):
    pool = [json.loads(x) for x in open(pool_path) if x.strip()]; random.shuffle(pool)
    reh = [p["messages"] for p in pool[:int(len(mix)*REPLAY_FRACTION)] if "messages" in p]
    print(f"replay: added {len(reh)} rehearsal rows")
else:
    print("replay: no rehearsal pool; train_mix == train_synth")
mixed = mix + reh; random.shuffle(mixed)
with open(f"{TASK}/train_mix.jsonl","w") as f:
    for m in mixed: f.write(json.dumps({"messages":m})+"\n")
print(f"wrote labels={len(labels)} gold={len(gold)} train_synth={len(seed_rows)} train_mix={len(mixed)}")

In [ ]:
#Config-driven training (adaptive length: clean-eval two-axis early stopping)
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainerCallback
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

_PROBES = [("sentinel","sentinel.jsonl","any",32),
           ("reasoning","reasoning_probes.jsonl","any",256),
           ("tools","tool_probes.jsonl","all",128)]

class TwoAxisEarlyStopping(TrainerCallback):
    def __init__(self, tok, out, es):
        self.tok, self.out = tok, out
        self.patience = es.get("patience", 2); self.min_delta = es.get("min_task_delta", 0.005)
        self.tol = es.get("regression_tolerance", CFG["acceptance"]["max_regression_drop"])
        self.batch = es.get("eval_batch", 8); lim = es.get("eval_task_limit", 0)
        gold = read_jsonl(f"{TASK}/gold.jsonl"); self.gold = gold[:lim] if lim else gold
        self.probes = []
        for axis, fn, mode, mx in _PROBES:
            p = f"{PROBES}/{fn}"
            if os.path.exists(p):
                rws = read_jsonl(p)
                self.probes.append((axis, [[{"role":"user","content":r["question"]}] for r in rws], rws, mode, mx))
        self.history, self.peaks = [], {}
        self.best_task, self.best_epoch, self.bad, self.stop_reason = float("-inf"), None, 0, None
    def _score(self, adapter_dir):
        m, tk = _load(adapter_dir)
        try:
            raw = _gen(m, tk, [r["messages"] for r in self.gold], 384, self.batch)
            task = macro_f1([r["label"] for r in self.gold], [c_predict_label(o, LABELS) for o in raw], LABELS)
            sc = {axis: score_probes(_gen(m, tk, pr, mx, self.batch), rws, mode)
                  for axis, pr, rws, mode, mx in self.probes}
        finally:
            del m; torch.cuda.empty_cache()
        return task, sc
    def on_epoch_end(self, args, state, control, **kw):
        model = kw.get("model")
        if model is None: return control
        epoch = round(state.epoch or 0); tmp = f"{self.out}/_epoch_eval"
        try:
            model.save_pretrained(tmp); task, sc = self._score(tmp)
        except Exception as e:
            print(f"[early-stop] epoch {epoch}: eval skipped ({type(e).__name__}: {e})"); return control
        reg_drop = max([self.peaks.get(a,v)-v for a,v in sc.items()] or [0.0]); reg_ok = reg_drop <= self.tol + 1e-9   # epsilon: exactly-at-tolerance drop is tolerated (float edge)
        self.history.append({"epoch":epoch,"task":round(task,4),**{k:round(v,4) for k,v in sc.items()}})
        print(f"[early-stop] epoch {epoch}: task={task:.3f} "+" ".join(f"{k}={v:.3f}" for k,v in sc.items())
              +f" | reg_drop_from_peak={reg_drop:.3f} (tol {self.tol:.3f})")
        if reg_ok and task > self.best_task + self.min_delta:
            self.best_task, self.best_epoch, self.bad = task, epoch, 0
            model.save_pretrained(self.out); self.tok.save_pretrained(self.out)
            print(f"[early-stop] new best task={task:.3f}; kept adapter from epoch {epoch}")
        else: self.bad += 1
        for a,v in sc.items(): self.peaks[a] = max(self.peaks.get(a,v), v)
        if not reg_ok: self.stop_reason = f"regression axis fell {reg_drop:.3f} > tol {self.tol:.3f} at epoch {epoch}"
        elif self.bad >= self.patience: self.stop_reason = f"task macro-F1 plateaued ({self.bad} eval(s)) at epoch {epoch}"
        if self.stop_reason:
            print(f"[early-stop] stopping: {self.stop_reason}; best epoch={self.best_epoch}")
            control.should_training_stop = True
        return control

def train_one(data_file, run_name, lr=None, epochs=None, seed=0):
    t, lo = CFG["train"], CFG["lora"]
    lr = t["lr"] if lr is None else lr; epochs = t["epochs"] if epochs is None else epochs
    es = t.get("early_stopping", {}); es_on = bool(es.get("enabled", False))
    tok = AutoTokenizer.from_pretrained(CFG["base_model"])
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(CFG["base_model"], quantization_config=quant,
        torch_dtype=torch.bfloat16, device_map="auto"); model.config.use_cache = False
    lora = LoraConfig(r=lo["rank"], lora_alpha=lo["alpha"], lora_dropout=lo["dropout"],
        bias="none", task_type="CAUSAL_LM", target_modules=lo["target_modules"])
    ds = load_dataset("json", data_files=f"{TASK}/{data_file}", split="train")
    out = f"{OUT}/{run_name}"
    cfg = SFTConfig(output_dir=out, num_train_epochs=epochs, per_device_train_batch_size=t["batch"],
        gradient_accumulation_steps=t["grad_accum"], learning_rate=lr, lr_scheduler_type="cosine",
        warmup_ratio=0.05, logging_steps=10, save_strategy=("no" if es_on else "epoch"), bf16=True,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        max_length=t["max_len"], packing=False, assistant_only_loss=True, seed=seed, report_to="none")
    stopper = TwoAxisEarlyStopping(tok, out, es) if es_on else None
    if es_on:
        print(f"[train] {run_name}: adaptive length on (max epochs={epochs}, patience={stopper.patience}, reg_tol={stopper.tol})")
    tr = SFTTrainer(model=model, args=cfg, train_dataset=ds, peft_config=lora, processing_class=tok,
                    callbacks=[stopper] if stopper else [])
    r = tr.train()
    if not (stopper and stopper.best_epoch is not None):
        tr.save_model(out); tok.save_pretrained(out)
    tail = (f" best_epoch={stopper.best_epoch}/{epochs} stop={stopper.stop_reason or 'reached max epochs'}" if stopper else "")
    print(f"[train] {run_name}: loss={r.training_loss:.4f}{tail} (data={data_file}, lr={lr}, seed={seed}, max_epochs={epochs})")
    return out

In [ ]:
#Two-axis evaluation (task gold from TASK, regression probes from PROBES)
def _load(adapter=None):
    tok = AutoTokenizer.from_pretrained(CFG["base_model"]); tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(CFG["base_model"], torch_dtype=torch.bfloat16)
    if adapter:
        from peft import PeftModel; m = PeftModel.from_pretrained(m, adapter)
    return m.to("cuda").eval(), tok

@torch.no_grad()
def _gen(m, tok, prompts, max_new, batch=16):
    outs = []
    for i in range(0, len(prompts), batch):
        ch = prompts[i:i+batch]
        txt = [tok.apply_chat_template(x, add_generation_prompt=True, tokenize=False) for x in ch]
        enc = tok(txt, return_tensors="pt", padding=True, add_special_tokens=False).to(m.device)
        g = m.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        for j in range(len(ch)):
            outs.append(tok.decode(g[j][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    return outs

def _probe(m, tok, fname, mode, max_new):
    probes = read_jsonl(f"{PROBES}/{fname}")
    raw = _gen(m, tok, [[{"role":"user","content":p["question"]}] for p in probes], max_new)
    return {"n": len(probes), "score": score_probes(raw, probes, mode)}

def evaluate(name, adapter=None):
    gold = read_jsonl(f"{TASK}/gold.jsonl")
    m, tok = _load(adapter)
    raw = _gen(m, tok, [r["messages"] for r in gold], 384)
    g = [r["label"] for r in gold]; p = [c_predict_label(o, LABELS) for o in raw]
    task = {"macro_f1": macro_f1(g, p, LABELS),
            "accuracy": sum(1 for a,b in zip(g,p) if a==b)/len(g),
            "valid": sum(1 for x in p if x is not None)/len(g)}
    res = {"name": name, "task": task,
           "sentinel": _probe(m, tok, "sentinel.jsonl", "any", 32),
           "reasoning": _probe(m, tok, "reasoning_probes.jsonl", "any", 256),
           "tools": _probe(m, tok, "tool_probes.jsonl", "all", 128)}
    json.dump(res, open(f"{OUT}/result_{name}.json","w"), indent=2)
    print(f"[eval] {name}: macroF1={task['macro_f1']:.3f} sentinel={res['sentinel']['score']:.3f} "
          f"reasoning={res['reasoning']['score']:.3f} tools={res['tools']['score']:.3f}")
    del m; torch.cuda.empty_cache(); return res

In [ ]:
#The gate + the orchestrated loop (median across a couple of seeds)
import statistics

def gate(base, cand):
    a = CFG["acceptance"]
    task_gain = cand["task"]["macro_f1"] - base["task"]["macro_f1"]
    worst = max((base[k]["score"] - cand[k]["score"]) for k in ["sentinel","reasoning","tools"])
    accept = task_gain >= a["min_task_gain_macro_f1"] and worst <= a["max_regression_drop"]
    print(f"[gate] task_gain={task_gain:+.3f} (min {a['min_task_gain_macro_f1']:+.3f}) | "
          f"worst_drop={worst:.3f} (max {a['max_regression_drop']:.3f}) -> {'ACCEPT' if accept else 'REJECT'}")
    return accept, task_gain, worst

def run_seeds(base, base_name, data_file, lr=None, epochs=None):
    seeds = CFG["train"].get("seeds", [0]); runs = []
    for s in seeds:
        name_s = f"{base_name}-s{s}" if len(seeds) > 1 else base_name
        adapter = train_one(data_file, name_s, lr=lr, epochs=epochs, seed=s)
        cand = evaluate(name_s, adapter); accept, gain, drop = gate(base, cand)
        runs.append({"seed": s, "name": name_s, "adapter": adapter, "accept": accept, "gain": gain, "drop": drop})
    return runs

def aggregate(runs):
    a = CFG["acceptance"]
    gains = [r["gain"] for r in runs]; drops = [r["drop"] for r in runs]
    mg, md = statistics.median(gains), statistics.median(drops)
    accept = mg >= a["min_task_gain_macro_f1"] and md <= a["max_regression_drop"]
    passing = [r for r in runs if r["accept"]]; chosen = max(passing or runs, key=lambda r: r["gain"])
    print(f"[gate] across seeds: {'ACCEPT' if accept else 'REJECT'} (median gain={mg:+.3f}, median drop={md:.3f}, "
          f"passed={len(passing)}/{len(runs)}, gain range=[{min(gains):+.3f}, {max(gains):+.3f}]) -> chosen {chosen['name']}")
    return accept, chosen

base = evaluate("base")
replay = CFG["guardrails"]["replay_mix"]
name1 = f"{CFG['name']}-{'replay' if replay else 'noreplay'}"
runs1 = run_seeds(base, name1, "train_mix.jsonl" if replay else "train_synth.jsonl")
ok1, chosen1 = aggregate(runs1)
final = chosen1["adapter"] if ok1 else None

if not ok1:
    adj = CFG["adjust_on_fail"]; lr = CFG["train"]["lr"] * adj["lower_lr_factor"]
    print(f"\n[pipeline] gate rejected; adjusted re-run (replay on, lr={lr}, epochs={adj['reduce_epochs_to']})")
    runs2 = run_seeds(base, f"{CFG['name']}-adj", "train_mix.jsonl", lr=lr, epochs=adj["reduce_epochs_to"])
    ok2, chosen2 = aggregate(runs2); final = chosen2["adapter"] if ok2 else None
print("\nACCEPTED:", final or "none")

In [ ]:
#Download adapters + results
import shutil, glob
made = []
for d in sorted(glob.glob(f"{OUT}/*/")):
    if os.path.exists(os.path.join(d, "adapter_config.json")):
        base_dir = d.rstrip("/"); shutil.make_archive(base_dir, "zip", base_dir); made.append(os.path.basename(base_dir))
print("zipped adapters:", made)
print("Download the lora *.zip and result_*.json from the Output tab.")